# Capstone: Skin lesion triage

7-class: classify dermatoscopic images of skin lesions (including melanoma).

This notebook gives you a **working baseline**. Your job is to make it better and
understand what you did. Look for the `# IMPROVE ME` markers. Use Claude freely as your
pair programmer, but be ready to explain every change you make.

**Remember the rubric:** build it, evaluate it honestly, find a failure mode, defend one
design decision, and make sure both partners can explain the work.

In [ ]:
# Setup: on Colab, grab the course files. Locally this is a no-op.
import os, sys
if not os.path.exists("../../capstone_common.py"):
    os.system("git clone -q https://github.com/jinchiwei/outset-ai-healthcare.git")
    os.chdir("outset-ai-healthcare/notebooks/day3_capstone/starter_kits/skin")
sys.path.insert(0, "../..")            # day3_capstone (for capstone_common)
sys.path.insert(0, "../../../_shared") # colab_setup
import colab_setup
colab_setup.ensure(*colab_setup.DAY1, "medmnist")

In [ ]:
import torch
sys.path.insert(0, "../..")
import capstone_common as cc

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

train_loader, val_loader, test_loader, n_classes, task = cc.get_loaders("dermamnist", size=64)
print("classes:", n_classes, "| task:", task)

## A look at the data

In [ ]:
import matplotlib.pyplot as plt
imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
for ax, im, lb in zip(axes, imgs, labels):
    ax.imshow(im[0], cmap="gray"); ax.set_title(int(lb)); ax.axis("off")
plt.tight_layout()

## Baseline: transfer learning (frozen ResNet18 + new head)

In [ ]:
model = cc.make_baseline(n_classes)
model = cc.train(model, train_loader, val_loader, epochs=3, lr=1e-3, device=device)

test_acc = cc.evaluate(model, test_loader, device=device)["accuracy"]
print(f"\nbaseline TEST accuracy: {test_acc:.3f}")

## # IMPROVE ME

The baseline freezes the whole ResNet and trains only the last layer for 3 epochs.
Real, easy wins to try (pick a few, with Claude's help):

- **Train longer**: more epochs.
- **Unfreeze the backbone**: let more of the ResNet learn (lower the learning rate if you do).
- **Augment**: add flips/rotations in the transform (see `capstone_common.get_loaders`).
- **Bigger input**: try `size=128` or `size=224`.
- **Class weighting**: if the classes are imbalanced, weight the loss.
- **Different model**: try resnet50, or a timm model.

After each change, re-check TEST accuracy. Did it actually help, or did you just overfit?

## Find a failure mode

Pull some test images your model got **wrong** and look at them. Is there a pattern?
Would a doctor have gotten them wrong too? This is rubric point 3, and often the most
interesting thing you'll present.

In [ ]:
import numpy as np
res = cc.evaluate(model, test_loader, device=device)
wrong = np.where(res["y"] != res["pred"])[0][:6]
print("indices the model got wrong:", wrong.tolist())
# IMPROVE ME: show these images with their true vs predicted labels, and write down
# what you notice.